# Multilingual RAG with BGE-M3, ChromaDB, and DeepSeek

This notebook demonstrates how to build a multilingual RAG pipeline using:
- **BGE-M3** — a powerful multilingual embedding model supporting 100+ languages, dense/sparse/multi-vector retrieval, and up to 8192 token sequences
- **ChromaDB** — a lightweight persistent vector store
- **DeepSeek** — as the LLM for answer generation via OpenAI-compatible API
- **LlamaIndex** — as the RAG orchestration framework

Unlike the standard `bge-base-en` example, BGE-M3 handles **mixed-language documents** — you can index English and Chinese text together and query in either language.

## Installation

In [ ]:
%pip install llama-index
%pip install llama-index-vector-stores-chroma
%pip install llama-index-embeddings-huggingface
%pip install llama-index-llms-openai-like
%pip install chromadb
%pip install FlagEmbedding

## Setup

In [ ]:
import os
import chromadb
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai_like import OpenAILike
from IPython.display import Markdown, display

## Configure BGE-M3 Embedding Model

BGE-M3 is a multilingual embedding model from BAAI that supports:
- 100+ languages including English, Chinese, Japanese, Korean, and more
- Up to 8192 token sequences (much longer than most models)
- Dense, sparse, and multi-vector retrieval modes

In [ ]:
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    max_length=512,
)

print(f"Embedding model loaded: {embed_model.model_name}")

## Configure DeepSeek LLM

DeepSeek provides an OpenAI-compatible API. Get your API key from https://platform.deepseek.com/

In [ ]:
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "your-api-key-here")

llm = OpenAILike(
    model="deepseek-chat",
    api_base="https://api.deepseek.com/v1",
    api_key=DEEPSEEK_API_KEY,
    is_chat_model=True,
)

# Set global defaults
Settings.embed_model = embed_model
Settings.llm = llm
Settings.chunk_size = 512
Settings.chunk_overlap = 50

## Create Multilingual Documents

We create a small multilingual dataset mixing English and Chinese text.
BGE-M3 embeds both languages into the same vector space, enabling cross-lingual retrieval.

In [ ]:
documents = [
    # English documents
    Document(
        text="""Artificial intelligence (AI) is transforming industries worldwide. 
        Machine learning, a subset of AI, enables systems to learn from data and improve 
        over time without being explicitly programmed. Deep learning uses neural networks 
        with many layers to model complex patterns in data.""",
        metadata={"language": "en", "topic": "AI overview"}
    ),
    Document(
        text="""Large language models (LLMs) are trained on vast amounts of text data 
        using self-supervised learning. Models like GPT, LLaMA, and DeepSeek use the 
        transformer architecture with attention mechanisms to understand and generate 
        human-like text.""",
        metadata={"language": "en", "topic": "LLMs"}
    ),
    Document(
        text="""Retrieval-Augmented Generation (RAG) combines information retrieval with 
        language generation. It first retrieves relevant documents from a knowledge base, 
        then uses an LLM to generate answers grounded in those documents. This reduces 
        hallucinations and keeps responses factual.""",
        metadata={"language": "en", "topic": "RAG"}
    ),
    # Chinese documents
    Document(
        text="""人工智能（AI）正在全球范围内改变各行各业。
        机器学习是人工智能的一个子集，它使系统能够从数据中学习，
        并随着时间的推移不断改进，而无需明确编程。
        深度学习使用多层神经网络来对数据中的复杂模式进行建模。""",
        metadata={"language": "zh", "topic": "AI概述"}
    ),
    Document(
        text="""大型语言模型（LLM）使用自监督学习在大量文本数据上进行训练。
        GPT、LLaMA和DeepSeek等模型使用具有注意力机制的Transformer架构
        来理解和生成类人文本。DeepSeek是由中国公司深度求索开发的先进语言模型。""",
        metadata={"language": "zh", "topic": "大语言模型"}
    ),
    Document(
        text="""检索增强生成（RAG）将信息检索与语言生成相结合。
        它首先从知识库中检索相关文档，然后使用大型语言模型生成基于这些文档的答案。
        这种方法可以减少幻觉现象，使回答更加真实可靠。""",
        metadata={"language": "zh", "topic": "检索增强生成"}
    ),
]

print(f"Created {len(documents)} documents ({sum(1 for d in documents if d.metadata['language']=='en')} English, {sum(1 for d in documents if d.metadata['language']=='zh')} Chinese)")

## Set Up ChromaDB Vector Store

In [ ]:
# Use persistent client to save embeddings to disk
chroma_client = chromadb.PersistentClient(path="./chroma_bge_m3")
chroma_collection = chroma_client.get_or_create_collection("multilingual_rag")

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print(f"ChromaDB collection: {chroma_collection.name}")
print(f"Existing documents: {chroma_collection.count()}")

## Build the Index

LlamaIndex will embed all documents using BGE-M3 and store them in ChromaDB.

In [ ]:
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True,
)

print(f"\nIndex built. Total vectors in ChromaDB: {chroma_collection.count()}")

## Query in English

BGE-M3 retrieves relevant documents regardless of the query language.

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=3)

response = query_engine.query("What is Retrieval-Augmented Generation and how does it work?")
display(Markdown(f"**Query (EN):** What is Retrieval-Augmented Generation and how does it work?\n\n**Answer:** {response}"))

## Query in Chinese

The same index handles Chinese queries — BGE-M3 maps both languages to the same embedding space.

In [ ]:
response_zh = query_engine.query("什么是大型语言模型？它们是如何工作的？")
display(Markdown(f"**查询 (ZH):** 什么是大型语言模型？它们是如何工作的？\n\n**回答:** {response_zh}"))

## Cross-lingual Query

Query in English but retrieve Chinese documents (and vice versa) — this is the unique power of BGE-M3.

In [ ]:
# Query in English, expect retrieval from both English and Chinese docs
response_cross = query_engine.query("How do neural networks learn from data?")
display(Markdown(f"**Cross-lingual Query:** How do neural networks learn from data?\n\n**Answer:** {response_cross}"))

# Show which source documents were retrieved
print("\nRetrieved source documents:")
for i, node in enumerate(response_cross.source_nodes):
    lang = node.metadata.get('language', 'unknown')
    topic = node.metadata.get('topic', 'unknown')
    score = node.score
    print(f"  [{i+1}] language={lang}, topic={topic}, score={score:.4f}")

## Load Existing Index from ChromaDB

Since we used `PersistentClient`, embeddings are saved to disk.
You can reload the index without re-embedding documents.

In [ ]:
# Reload from disk — no re-embedding needed
chroma_client_reload = chromadb.PersistentClient(path="./chroma_bge_m3")
chroma_collection_reload = chroma_client_reload.get_collection("multilingual_rag")
vector_store_reload = ChromaVectorStore(chroma_collection=chroma_collection_reload)

index_reload = VectorStoreIndex.from_vector_store(
    vector_store=vector_store_reload,
)

query_engine_reload = index_reload.as_query_engine(similarity_top_k=3)
response_reload = query_engine_reload.query("DeepSeek是什么模型？")
display(Markdown(f"**Reloaded index query:** DeepSeek是什么模型？\n\n**Answer:** {response_reload}"))

## Summary

This notebook demonstrated:

| Feature | Details |
|---|---|
| Embedding model | `BAAI/bge-m3` — multilingual, 8192 max tokens |
| Vector store | ChromaDB with persistent storage |
| LLM | DeepSeek via OpenAI-compatible API |
| Languages | English + Chinese in the same index |
| Cross-lingual retrieval | Query in EN → retrieve ZH docs (and vice versa) |

**Why BGE-M3 over bge-base-en?**
- Supports 100+ languages vs English-only
- 8192 token context vs 512
- Better performance on multilingual benchmarks
- Same simple HuggingFaceEmbedding interface